# Week 4 (authored from the course README spec — no helper notebook exists)

**Goals (from README):** define what a "good" latent representation is and evaluate it:
1. **PRED Y** — train a model on Ztrain to predict a downstream Y not in X; compare with the same model on raw X.
2. **X RECOVERY** — for each numerical feature, can Z recover the raw X_j?
3. **GEOMETRY** — do distances in latent space match distances in observation space? Inspect nearest neighbours.
4. **(Optional) PCA** — 2D PCA of Z coloured by Y / age.
5. At least one comparison across latent dim / encoding / hyperparameter (here: latent dim 2 vs 8 vs 32).

Setup reuses Week 2's frozen features + splits and Week 3's standardized-input VAE
(hidden 64, latent 8, beta 1.0, KL warmup 10, lr 1e-3, batch 256, 100 epochs, seed 42).

In [ ]:
import json, pickle as pk, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))  # repo root, for proj34_representation imports
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import DataLoader, TensorDataset

from proj34_representation.tabular_vae.vae import TabularVAE, vae_loss
from proj34_representation.tabular_vae.preprocessing import TabularPreprocessor
import copy

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
OUT = Path("../../../results/week4"); OUT.mkdir(parents=True, exist_ok=True)

feature_sets = pk.load(open("../../data/feature_names.pkl", "rb"))
patient_agg = pd.read_parquet("../../data/clean_dataset.parquet")
xc = json.load(open("../../data/x_cols.json"))
X_num, X_bin, X_cat, X_cols = xc["num"], xc["bin"], xc["cat"], xc["all"]
splits = json.load(open("../../data/splits.json"))
dfs = {k: patient_agg.iloc[v].copy() for k, v in splits.items()}

num_cols = [c for c in X_cols if c in feature_sets["labs"].tolist()
            + feature_sets["vitals"].tolist() + ["age", "admissionheight", "admissionweight"]]
cat_cols = [c for c in X_cols if c not in num_cols]  # binary + categorical (as in Week 3)

# --- identical preprocessing to Week 3: median imputation (train), 'Missing' cats
medians = dfs["train"][num_cols].apply(lambda s: pd.to_numeric(s.replace("> 89", 90), errors="coerce").median())
def impute(df):
    num = df[num_cols].copy()
    for c in num_cols:
        num[c] = pd.to_numeric(num[c].replace("> 89", 90), errors="coerce")
    num = num.fillna(medians)
    cat = df[cat_cols].copy()
    for c in cat_cols:
        cat[c] = cat[c].fillna("Missing")
    return pd.concat([num, cat], axis=1)[X_cols]
imp = {k: impute(df) for k, df in dfs.items()}
pre = TabularPreprocessor(cat_cols, num_cols, cont_encoders=None)
Xmat = {"train": pre.fit_transform(imp["train"])}
for k in ["val", "test"]:
    # drop rows with categories unseen in train, KEEPING the original index
    # (the package's drop_unseen resets the index, which would break Y alignment)
    keep = np.ones(len(imp[k]), dtype=bool)
    for c in cat_cols:
        keep &= imp[k][c].isin(pre.categories_[c]).to_numpy()
    if not keep.all():
        print(f"{k}: dropped {(~keep).sum()} row(s) with unseen categories")
    imp[k] = imp[k][keep]
    Xmat[k] = pre.transform(imp[k])
Y = {k: patient_agg.loc[imp[k].index, "mortality_at48h"].fillna(0).astype(int) for k in imp}
print("input_dim:", pre.input_dim_, "| cat_total:", pre.cat_total_, "| cont_total:", pre.cont_total_)
print({k: (Xmat[k].shape, int(Y[k].sum())) for k in Xmat})


In [ ]:
# --- Train the standardized-input VAE (Week 3 configuration, latent 8) ---
def run_epoch(model, loader, beta, optimizer=None):
    training = optimizer is not None
    model.train(training)
    sums = {"recon_cat": 0.0, "recon_cont": 0.0, "kl": 0.0}; n = 0
    with torch.set_grad_enabled(training):
        for (x,) in loader:
            x_hat, mu, logvar = model(x)
            loss, parts = vae_loss(x, x_hat, mu, logvar, model.cat_dims, beta)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            n += x.size(0)
            for nm in sums: sums[nm] += parts[nm] * x.size(0)
    return {nm: v / n for nm, v in sums.items()}

def train_vae(latent_dim, epochs, Xtr, Xva):
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = TabularVAE(input_dim=pre.input_dim_, cat_dims=pre.cat_dims_, hidden_dim=64, latent_dim=latent_dim)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    ltr = DataLoader(TensorDataset(torch.from_numpy(Xtr)), batch_size=256, shuffle=True)
    lva = DataLoader(TensorDataset(torch.from_numpy(Xva)), batch_size=256, shuffle=False)
    best = (float("inf"), None); hist = []
    for ep in range(1, epochs + 1):
        b = 1.0 * min(1.0, ep / 10)
        tp = run_epoch(m, ltr, b, opt); vp = run_epoch(m, lva, b)
        vl = vp["recon_cat"] + vp["recon_cont"] + vp["kl"]
        hist.append({"epoch": ep, "val_loss": vl, **{f"val_{k}": v for k, v in vp.items()}})
        if vl < best[0]: best = (vl, copy.deepcopy(m.state_dict()))
    m.load_state_dict(best[1])
    return m, pd.DataFrame(hist)

model, hist = train_vae(8, 100, Xmat["train"], Xmat["val"])
print("Best val loss (latent 8, 100 epochs):", round(hist.val_loss.min(), 3),
      "at epoch", int(hist.loc[hist.val_loss.idxmin(), "epoch"]))
print(hist.iloc[[0, 24, 49, 99]].round(3).to_string(index=False))
torch.save(model.state_dict(), OUT / "vae_standard_latent8.pt")


In [ ]:
# --- Generate Z = encoder mean mu(x) for every split; save Ztrain/Zval/Ztest ---
@torch.no_grad()
def get_mu(m, X):
    m.eval(); out = []
    for (xb,) in DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=512):
        _, mu, _ = m(xb); out.append(mu.numpy())
    return np.vstack(out)

Z = {k: get_mu(model, Xmat[k]) for k in Xmat}
np.savez(OUT / "Z_standard_latent8.npz",
         **{f"Z_{k}": Z[k] for k in Z},
         **{f"stayid_{k}": patient_agg.loc[imp[k].index, "patienthealthsystemstayid"].to_numpy() for k in Z})
print({k: Z[k].shape for k in Z})
print("Saved Z arrays + model to", OUT)


In [ ]:
# --- 1) PRED Y: logistic regression on Z vs on X (test set) ----------------
def pred_metrics(Ztr, ytr, Zte, yte):
    lr = LogisticRegression(max_iter=1000).fit(Ztr, ytr)
    p = lr.predict_proba(Zte)[:, 1]
    return roc_auc_score(yte, p), average_precision_score(yte, p)

rows = []
a, b = pred_metrics(Z["train"], Y["train"], Z["test"], Y["test"])
rows.append({"representation": "Z (VAE latent, dim 8)", "test_AUROC": round(a, 3), "test_AUPRC": round(b, 3)})
a, b = pred_metrics(Xmat["train"], Y["train"], Xmat["test"], Y["test"])
rows.append({"representation": "X (full preprocessed matrix)", "test_AUROC": round(a, 3), "test_AUPRC": round(b, 3)})
sc = StandardScaler().fit(imp["train"][num_cols])
a, b = pred_metrics(sc.transform(imp["train"][num_cols]), Y["train"],
                    sc.transform(imp["test"][num_cols]), Y["test"])
rows.append({"representation": "X numerical only (standardized)", "test_AUROC": round(a, 3), "test_AUPRC": round(b, 3)})
pred_table = pd.DataFrame(rows)
print(pred_table.to_string(index=False))
print(f"(test: n={len(Y['test'])}, deaths={int(Y['test'].sum())}, prevalence={Y['test'].mean():.3f})")


In [ ]:
# --- 2) X RECOVERY: per-feature linear regression Z -> raw X_j (test R^2) --
rec = []
for c in num_cols:
    lr = LinearRegression().fit(Z["train"], imp["train"][c])
    r2 = r2_score(imp["test"][c], lr.predict(Z["test"]))
    s = pd.to_numeric(patient_agg[c].replace("> 89", 90), errors="coerce")
    rec.append({"feature": c, "R2_from_Z": round(r2, 3),
                "skew": round(s.skew(), 2), "missing_%": round(100 * s.isna().mean(), 1)})
rec = pd.DataFrame(rec).sort_values("R2_from_Z", ascending=False)
print(rec.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(rec["feature"][::-1], rec["R2_from_Z"][::-1])
ax.set_xlabel("Test R2 of recovering raw feature from Z (linear)")
plt.tight_layout(); plt.savefig(OUT / "x_recovery.png", dpi=110); plt.show()


In [ ]:
# --- 3) GEOMETRY: are latent distances like observation distances? --------
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Z["test"]), size=min(300, len(Z["test"])), replace=False)
Xnum_test = sc.transform(imp["test"][num_cols])  # standardized numerical space
def pairwise(A):
    d = np.sqrt(((A[:, None, :] - A[None, :, :]) ** 2).sum(-1))
    return d[np.triu_indices(len(A), 1)]
dX, dZ = pairwise(Xnum_test[idx]), pairwise(Z["test"][idx])
pr = pearsonr(dX, dZ); sr = spearmanr(dX, dZ)
print(f"Pairwise-distance agreement on {len(idx)} test patients:")
print(f"  Pearson r = {pr[0]:.3f}   Spearman rho = {sr[0]:.3f}")
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(dX, dZ, s=2, alpha=0.3)
ax.set_xlabel("distance in X (standardized numerical)"); ax.set_ylabel("distance in Z")
ax.set_title(f"Geometry: Pearson r={pr[0]:.2f}, Spearman={sr[0]:.2f}")
plt.tight_layout(); plt.savefig(OUT / "geometry_scatter.png", dpi=110); plt.show()

# Nearest neighbours in Z for three example test patients
def show_neighbours(i, label):
    d = np.sqrt(((Z["test"] - Z["test"][i]) ** 2).sum(1)); nn = np.argsort(d)[1:6]
    base = imp["test"].iloc[i]
    nb = imp["test"].iloc[nn]
    print(f"\n{label}: age={base['age']:.0f}, mortality={int(Y['test'].iloc[i])}, "
          f"heartrate={base['heartrate_bin1']:.0f}, creatinine={base['creatinine']:.2f}")
    print(f"  5 nearest in Z: ages={list(nb['age'].astype(int))}, "
          f"mortality={list(Y['test'].iloc[nn])}, "
          f"heartrate mean={nb['heartrate_bin1'].mean():.0f}, creatinine mean={nb['creatinine'].mean():.2f}")
test_reset = imp["test"].reset_index(drop=True)
show_neighbours(0, "Patient A (first test row)")
died = np.flatnonzero(Y["test"].to_numpy() == 1)
if len(died): show_neighbours(int(died[0]), "Patient B (died 24-48h)")
show_neighbours(10, "Patient C (row 10)")


In [ ]:
# --- 4) PCA of Z (optional in the README) ----------------------------------
pca = PCA(n_components=2).fit(Z["train"])
Zte2 = pca.transform(Z["test"])
print("PCA explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(Zte2[:, 0], Zte2[:, 1], c=Y["test"], cmap="coolwarm", s=14)
axes[0].set_title("Z (test), PCA coloured by mortality_at48h")
sc2 = axes[1].scatter(Zte2[:, 0], Zte2[:, 1], c=imp["test"]["age"], cmap="viridis", s=14)
axes[1].set_title("Z (test), PCA coloured by age"); fig.colorbar(sc2, ax=axes[1])
plt.tight_layout(); plt.savefig(OUT / "pca_z.png", dpi=110); plt.show()


In [ ]:
# --- 5) Comparison across latent dims (README requires >= 1 comparison) ---
rows = []
for ld in [2, 8, 32]:
    m_ld, h_ld = (model, hist) if ld == 8 else train_vae(ld, 50, Xmat["train"], Xmat["val"])
    Ztr = get_mu(m_ld, Xmat["train"]); Zte = get_mu(m_ld, Xmat["test"])
    a, b = pred_metrics(Ztr, Y["train"], Zte, Y["test"])
    r2s = []
    for c in num_cols:
        lr = LinearRegression().fit(Ztr, imp["train"][c])
        r2s.append(r2_score(imp["test"][c], lr.predict(Zte)))
    rows.append({"latent_dim": ld, "best_val_loss": round(h_ld.val_loss.min(), 3),
                 "probe_test_AUROC": round(a, 3), "probe_test_AUPRC": round(b, 3),
                 "mean_recovery_R2": round(float(np.mean(r2s)), 3)})
    print(rows[-1])
dim_table = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(dim_table.latent_dim, dim_table.probe_test_AUROC, marker="o", label="probe AUROC (test)")
ax.plot(dim_table.latent_dim, dim_table.mean_recovery_R2, marker="s", label="mean X-recovery R2")
ax.set_xscale("log", base=2); ax.set_xticks([2, 8, 32]); ax.set_xticklabels([2, 8, 32])
ax.set_xlabel("latent dim"); ax.legend(); ax.set_title("Latent dim comparison (dims 2/32 trained 50 epochs)")
plt.tight_layout(); plt.savefig(OUT / "latent_dim_comparison.png", dpi=110); plt.show()


In [ ]:
# --- Save all Week 4 numbers ----------------------------------------------
summary = {
    "pred_y": pred_table.to_dict("records"),
    "x_recovery": rec.to_dict("records"),
    "geometry": {"pearson": float(pr[0]), "spearman": float(sr[0]), "n_patients": int(len(idx))},
    "latent_dims": dim_table.to_dict("records"),
    "vae_best_val_loss_latent8": float(hist.val_loss.min()),
}
json.dump(summary, open(OUT / "week4_metrics.json", "w"), indent=1)
print("Wrote", OUT / "week4_metrics.json")
